# Mathematical Foundations for Machine Learning

## 0 · One kick, a few useful questions

You're 20 metres from goal. Aim directly at a spot in the net and gravity will pull the ball below it. Aim higher and you might clear the wall, or the crossbar too. What would let you reason about the shot rather than keep guessing?

Start with a picture you can hold in your head: one ball, a forward-and-upward launch, and gravity continually bending its path downward. We'll ask four questions of that same kick.

| Question | What we'll use | What the notebook checks |
| --- | --- | --- |
| How much of the launch points toward the target? | Vectors and dot products | The signed shadow of one direction on another |
| Where is the ball heading right now? | Derivatives | Rising, level, and falling velocity readings |
| How do small moves become a whole flight? | Integration and matrices | 28 local updates agree with the direct formula |
| What if the player cannot repeat the angle exactly? | Probability | Expected scoring chances under an assumed spread |

You need arithmetic and a little patience with coordinate pairs, not prior calculus. This is a gravity-only model, not a trained football predictor. The rounded flight numbers below are calculations from that model, not observations of real kicks.

![Gravity-only free-kick trajectory assembled from repeated local updates, clearing the wall and reaching the target](images/gravity-only-trajectory.svg)

Read the cyan bar as the next forward displacement and the gold bar as the next upward or downward displacement. Both are **distances during one slice**, not velocities. Velocity is the distance-per-second reading used to calculate them.

Early gold bars point upward. Near the top they shrink, then point downward. The ball keeps travelling forward throughout. No individual bar describes the whole curve; adding the moves does.

## Set up the pitch before naming the maths

Put the launch point at coordinate `[0, 0]`. The first number is horizontal distance in metres; the second is the height of the ball's centre. Time starts at contact. This idealized launch starts at height zero; the later radius checks keep the ball clear of the wall and goal frame.

- Launch speed: `20 m/s`.
- Wall: `9.15 m` ahead, `1.8 m` high.
- Goal line: `20 m` ahead, crossbar at `2.44 m`.
- Chosen back-net target: `[21.5 m, 1.10 m]`.
- Ball radius: `0.11 m`.

That radius matters: the centre must be above `1.91 m` at the wall, and between `0.11 m` and `2.33 m` at the goal line. Reaching a point behind the goal and fitting through its front opening are different checks.

The setup cell gives these quantities short names. `g = 9.81` means vertical velocity drops by `9.81 m/s` every second; `v0` is launch speed. `ball_state(x, theta_deg)` asks for the time, height, and velocity at horizontal position `x` when launched at angle `theta_deg`.

We ignore wind, drag, and spin. With only constant gravity, horizontal velocity stays fixed and vertical velocity steadily decreases. That simple rule is enough to make a curved path.

In [ ]:
# Dependencies
import subprocess, sys

# Only install packages that are not already importable in this environment
required = [("numpy", "numpy"), ("scipy", "scipy")]
for imp, pkg in required:
    try:
        __import__(imp)
        print(f"  ok  {pkg}")
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"  done {pkg}")

import numpy as np
from scipy import stats

np.random.seed(42)
print("All dependencies ready")

# Physical constants for the free kick scenario
g = 9.81          # gravity changes downward velocity by 9.81 m/s each second
v0 = 20.0        # launch speed (m/s)
BALL_RADIUS = 0.11
WALL_X = 9.15    # wall position (m)
WALL_H = 1.8     # wall height (m)
GOAL_X = 20.0    # front of the goal (m)
CROSS_H = 2.44   # crossbar height (m)
NET_X = 21.5     # back of the net (m)
TARGET_H = 1.10  # intended ball-centre height at back-net impact (m)
GOAL_BOTTOM = BALL_RADIUS
GOAL_TOP = CROSS_H - BALL_RADIUS


def ball_state(x, theta_deg):
    """Return time, height, and velocity when the ball reaches horizontal position x."""
    theta = np.radians(theta_deg)
    vx = v0 * np.cos(theta)
    t = x / vx
    vy = v0 * np.sin(theta) - g * t
    y = v0 * np.sin(theta) * t - 0.5 * g * t**2
    return {"x": float(x), "t": float(t), "y": float(y), "vx": float(vx), "vy": float(vy)}


def ball_height(x, theta_deg):
    """Height of the ball centre at horizontal position x and launch angle theta (degrees)."""
    return ball_state(x, theta_deg)["y"]


print("\nFree kick setup:")
print(f"  Launch speed: {v0} m/s")
print(f"  Gravity changes downward velocity by {g} m/s each second")
print(f"  Wall at {WALL_X}m, ball centre must clear {WALL_H + BALL_RADIUS:.2f}m")
print(f"  Goal plane at {GOAL_X}m, legal centre window [{GOAL_BOTTOM:.2f}, {GOAL_TOP:.2f}]m")
print(f"  Back-net target at ({NET_X:.1f}m, {TARGET_H:.2f}m)")

## Part 1 · An arrow can point mostly toward a target and still miss it

At contact, the ball has one velocity vector: an arrow with direction and speed. A coordinate pair stores its forward and upward components. The target vector has a different job: it points from launch to the chosen finish, not in the direction the ball must initially travel.

The target rises only `1.10 m` over `21.5 m`, a shallow `2.93°` line. Gravity means the launch must point higher:

| Launch angle at 20 m/s | Height at horizontal distance 21.5 m | Reading |
| ---: | ---: | --- |
| `15°` | about `-0.31 m` | The mathematical path is already below ground |
| `19.11°` | about `1.10 m` | Reaches our chosen target |
| `25°` | about `3.12 m` | Passes above the target |

These are rounded model calculations. The code finds the lower working angle with `brentq`: try angles inside a bracket until predicted height minus target height is zero. It solves known physics; it does not learn from kicks.

Now ask a narrower question: **how much of that initial velocity points along the direct target line?**

![Dot product visualized as the signed shadow of the kick arrow along the direct target line](images/free-kick-vector-projection.svg)

The green shadow is the velocity projected onto that line. Both arrows can first be scaled to length one, so their comparison reports direction without speed dominating it. The angle gap is about `16.18°`; its shadow fraction is about `0.9604`.

> **Visual to add — A shadow is not a scoring chance**
>
> Draw the 20 m/s launch arrow at 19.11°, the shallow 2.93° target rail, and a perpendicular drop from the arrow tip onto the rail. Mark the projected length 19.21 m/s; beside it show the later curved flight. Contrast initial alignment with eventual landing. Do not draw an arrow from “96% alignment” to “96% chance of scoring”: the shadow measures neither probability nor energy transferred.
>
> **Proposed caption:** “The dot product reads one direction now. Gravity determines what happens next.”

### Give the shadow a sign

Keep the launch direction fixed and rotate the rail you care about.

- Same direction: a unit arrow casts a shadow of `+1`.
- Perpendicular: its shadow is `0`.
- Opposite direction: its signed shadow is `-1`.

That signed shadow is a **dot product**. For unit arrows it lies between `-1` and `1`. For a real velocity projected onto a unit direction, it has units of speed.

Here the unit-arrow dot product is `0.9604`. Multiplying by `20 m/s` gives about `19.21 m/s` along the target rail. It is not “96% of the kick's power”: energy and force are different quantities.

**Predict:** if you turn the launch 90° *relative to the target rail*, should the score be zero or one? Zero. An absolute launch angle of 90° would be perpendicular to the ground, not quite perpendicular to our tilted rail.

### How the computer reads the arrows

A direction at `19.11°` becomes approximately `[0.945, 0.327]`: mostly forward, somewhat upward. The target direction becomes `[0.999, 0.051]`. Both pairs have length one before rounding.

Match forward with forward, upward with upward, then add:

```text
[0.945, 0.327] dot [0.999, 0.051]
    0.945 * 0.999 + 0.327 * 0.051 ≈ 0.960
```

The rounded arithmetic explains the operation; the code uses unrounded values. In notation, `a dot b = a_x*b_x + a_y*b_y`: subscripts simply label the two matching coordinate directions.

An array shape of `(2,)` means two entries in one vector, not two examples. The dot product reduces those matching entries to **one scalar**.

### Read the next cell as three moves

1. Solve for an angle whose calculated back-net height is `1.10 m`.
2. Use sine and cosine to split each direction into its two coordinates.
3. Dot the unit directions and convert the result back into an angle gap.

`arctan2(rise, run)` finds the target's direction. `cos` and `sin` give an arrow's forward and upward fractions. NumPy takes angles in radians; the degree conversions keep the printed numbers familiar. Clipping the dot product to `[-1, 1]` guards `arccos` against tiny rounding overshoots.

In [ ]:
# Part 1: solve the pictured geometry, then compare the two directions
from scipy.optimize import brentq

# The direct target angle comes from the rise-to-run ratio explained above.
target_line_deg = np.degrees(np.arctan2(TARGET_H, NET_X))

# Find the lower angle where predicted target height minus intended height is zero.
def target_height_error(angle_deg):
    return ball_height(NET_X, angle_deg) - TARGET_H


launch_angle_deg = brentq(target_height_error, 5.0, 45.0)
assert np.isclose(target_height_error(launch_angle_deg), 0.0)

# Each (2,) arrow keeps horizontal and vertical components together;
# unit length lets us compare direction without launch speed changing the score.
kick_dir = np.array([
    np.cos(np.radians(launch_angle_deg)),
    np.sin(np.radians(launch_angle_deg)),
])
target_dir = np.array([
    np.cos(np.radians(target_line_deg)),
    np.sin(np.radians(target_line_deg)),
])

# (2,) dot (2,) -> scalar: pair matching directions, then sum their agreement.
# Unit arrows make this a signed shadow fraction rather than a speed.
shadow_score = np.dot(kick_dir, target_dir)
angle_between = np.degrees(np.arccos(np.clip(shadow_score, -1, 1)))

print(f"Kick direction ({launch_angle_deg:.2f}°):   {kick_dir.round(3)}")
print(f"Target direction ({target_line_deg:.2f}°): {target_dir.round(3)}")
print(f"Angle between arrows: {angle_between:.2f}°")
print(f"Forward-shadow score: {shadow_score:.4f} out of 1")
print()
print("Interpretation: most of the launch direction points along the direct target line.")

### That answers alignment, not motion

The next question is not “is 0.9604 large enough?” It is “what happens after gravity has acted for half a second?”

The launch and target arrows need not stay aligned with the ball's current velocity. The target stays put; the velocity turns downward.

**Small experiment:** in the direction-building code, compare the target rail with a direction `target_line_deg + 90`, then with `target_line_deg + 180`. Keep the target fixed. The dot products should be near zero and minus one. This tests the sign of a shadow, not whether either kick reaches the goal.

## Part 2 · Freeze the flight: which way is it moving now?

At `t = 0.50 s`, the model puts the ball near `[9.45, 2.05] m`. Its velocity is about `[18.90, +1.64] m/s`. It is still rising, but only just.

Imagine gravity switching off at that instant. The ball would follow that velocity arrow along a straight tangent to the curve. The tangent describes **motion now**, not the whole future under gravity. This instantaneous change-per-time is the **derivative** of position.

In the usual shorthand, `v_x = dx/dt` and `v_y = dy/dt`. Read `dx/dt` as “how quickly horizontal position changes as time passes,” not as a new object to memorize.

Over the next illustrative `0.04 s`, that frozen velocity predicts:

```text
forward: 18.90 * 0.04 ≈ 0.756 m
upward:   1.64 * 0.04 ≈ 0.066 m
gravity correction: -0.5 * 9.81 * 0.04² ≈ -0.008 m
net rise: about 0.058 m
```

Meanwhile vertical speed drops by `9.81 * 0.04 ≈ 0.392 m/s`. A change in **height** and a change in **velocity** have different units; don't add them to each other.

![One exact gravity-only update showing the horizontal move, vertical move, and gravity correction at step 10](images/gravity-only-step.svg)

The existing illustration shows step 10, not the half-second frame. Its slice is about `0.04063 s`, and its starting vertical speed is different. There the upward contribution is about `0.120 m`, gravity removes `0.008 m`, and the rise is about `0.112 m`.

> **Visual to add — Two clocks in one step**
>
> Show the half-second state card `[height 2.047 m, upward speed 1.642 m/s]` beside a 0.04-second interval. Draw a 0.0657 m upward displacement and a 0.00785 m downward correction; separately lower the speedometer by 0.3924 m/s. Contrast position change with speed change. Do not connect “-0.3924 m/s” directly to the height axis as though it were a distance.
>
> **Proposed caption:** “Velocity moves the ball; acceleration changes the velocity.”

The next cell repeats the same move 28 times: add `vx * delta_t` to horizontal position, add `vy * delta_t - 0.5*g*delta_t**2` to height, then subtract `g*delta_t` from vertical velocity.

Because gravity is constant, that within-slice correction is exact for this model. The cell is not using the rough “freeze velocity and ignore acceleration” approximation.

In [ ]:
# Part 2: repeat the gravity-only update derived above
theta_deg = launch_angle_deg
theta_rad = np.radians(theta_deg)
step_count = 28

velocity_x = v0 * np.cos(theta_rad)
position_x = 0.0
position_y = 0.0
velocity_y = v0 * np.sin(theta_rad)
flight_time = NET_X / velocity_x
delta_t = flight_time / step_count

step_rows = []
for step_index in range(step_count):
    delta_x = velocity_x * delta_t
    delta_y = velocity_y * delta_t - 0.5 * g * delta_t**2
    next_velocity_y = velocity_y - g * delta_t

    if step_index in {0, 9, 18, 27}:
        step_rows.append((step_index + 1, position_x, position_y, delta_x, delta_y, next_velocity_y))

    position_x += delta_x
    position_y += delta_y
    velocity_y = next_velocity_y

# Compare the accumulated small steps with the direct projectile formula and target.
net_state = ball_state(NET_X, theta_deg)
step_error = np.hypot(position_x - net_state["x"], position_y - net_state["y"])
target_error = abs(net_state["y"] - TARGET_H)

print(f"28 slices across {flight_time:.3f}s give Δt={delta_t:.3f}s\n")
print(" step   start (x, y)       Δx        Δy       next vy")
for step_number, start_x, start_y, change_x, change_y, next_vy in step_rows:
    print(
        f" {step_number:>2d}    ({start_x:>5.2f}, {start_y:>4.2f})   "
        f"{change_x:+.3f}m   {change_y:+.3f}m   {next_vy:+.3f}m/s"
    )

print(f"\nAccumulated endpoint: ({position_x:.6f}, {position_y:.6f}) m")
print(f"Direct-formula endpoint: ({net_state['x']:.6f}, {net_state['y']:.6f}) m")
print(f"Intended target: ({NET_X:.6f}, {TARGET_H:.6f}) m")
print(f"Endpoint disagreement: {step_error:.2e} m")
print(f"Target-height error: {target_error:.2e} m")
assert step_error < 1e-10
assert target_error < 1e-10

x_peak_analytic = v0**2 * np.sin(2 * theta_rad) / (2 * g)
peak_state = ball_state(x_peak_analytic, theta_deg)
wall_state = ball_state(WALL_X, theta_deg)
height_left_after_wall = peak_state["y"] - wall_state["y"]
print(f"\nWall clearance: x={WALL_X:.2f}m, y={wall_state['y']:.2f}m, vy={wall_state['vy']:+.2f}m/s")
print(f"Turning point: x={x_peak_analytic:.2f}m, y={peak_state['y']:.2f}m, vy={peak_state['vy']:+.2e}m/s")
print(f"Height still to rise after the wall: {height_left_after_wall:.2f}m")
print("Prediction check: answer 2 — wall clearance happens near the top of the arc.")

### Read the check, not just the endpoint

`step_rows` saves four snapshots before each selected move, so “start (x, y)” is the old state and “next vy” belongs to the new state. The endpoint check compares all 28 accumulated moves with `ball_state`'s direct formula; both must reach `[21.5, 1.10] m` within floating-point tolerance.

At the wall, height is about `2.02 m` and vertical velocity about `+1.80 m/s`. The ball rises roughly `0.16 m` more to its `2.18 m` peak. At that point vertical velocity is zero, **not the whole velocity**: horizontal motion continues.

**Predict:** after the peak, does height become negative immediately? No. The vertical *velocity* becomes negative first. The ball can be above the ground while moving downward.

## Stitch the moves together: integration

One speedometer reading gives one local move. Add all the successive moves and you recover the flight. That accumulation is **integration**.

![Animation showing 4, 8, 16, and 28 repeated local predictions becoming a progressively smoother free-kick curve](images/calculus-accumulation.gif)

More frames make the drawn path smoother. They do not improve the endpoint in our constant-gravity implementation: each slice already includes the exact gravity correction.

The printed 28-step snapshots have these rounded moves:

| Step | Forward distance | Vertical distance |
| ---: | ---: | ---: |
| 1 | `+0.768 m` | `+0.258 m` |
| 10 | `+0.768 m` | `+0.112 m` |
| 19 | `+0.768 m` | `-0.034 m` |
| 28 | `+0.768 m` | `-0.179 m` |

All 28 moves, not just these four, sum to the final displacement. Upward and downward pieces partially cancel.

If we used only `vertical speed * slice duration`, each slice would be an approximation. Making those slices thinner gives the familiar integral `integral of v_y over time`: the signed area under the vertical-velocity curve. Its unit is `(metres/second) * seconds = metres`.

### Change and accumulation are the two views of one journey

- A derivative asks, “how fast is this quantity changing **here**?”
- An integral asks, “what change accumulated **over this interval**?”

For this kick, position is the quantity and time is what we vary. In ML, you will often vary a **weight** and ask how the **loss** responds. That derivative has a different meaning and units; it is not a physical velocity.

The next notebook will use local changes to reason about fitted predictions. First, how do we package several update rules so they cannot lose track of each other?

## Part 3 · A matrix is a stack of recipes

Keep a two-entry state card: `[height, vertical speed]`. Using rounded illustrative values from the half-second frame, it reads `[2.05, 1.64]`.

To advance it by `0.04 s`, one recipe reads “keep height and add 0.04 times speed.” Another reads “keep speed.” Put their coefficients into rows:

```text
                  state now             without gravity
[ 1   0.04 ]     [ 2.05 ]               [ 2.1156 ]
[ 0   1    ]  @  [ 1.64 ]      =        [ 1.6400 ]
   (2 x 2)         (2 x 1)                  (2 x 1)

add gravity's two corrections: [-0.007848 m, -0.3924 m/s]
state next:                    [ 2.107752 m, 1.2476 m/s]
```

Each row produces one output. The first column says how much old height to use; the second says how much old speed to use. The entries need not all have the same units: `0.04` carries seconds so the first row produces a height.

This is **matrix multiplication**: several dot products sharing one input. Adding the gravity vector makes the whole update an *affine* transformation, meaning a linear mix plus an offset.

This numerical state-card calculation explains the packaging; the preceding executable loop implements the same physics as separate scalar updates.

### Same operation, different source of the numbers

Physics supplied those coefficients. In a learned layer, training chooses coefficients that help a prediction instead.

That does **not** mean every matrix is a neural network, or every pair of numbers is a physical state. Always ask what the rows and columns represent. The multiplication stays the same when the meaning changes.

### Let two recipes read the same kick

The next cell switches from physical state to a made-up feature card `[angle signal, speed signal] = [0.4, 0.8]`. These are illustrative, dimensionless inputs; the cell does not calculate them from the solved launch angle or speed.

The weight table `W` has shape `(2, 2)`: **output recipes by input features**. The vector `b` has shape `(2,)`: one offset per output. Neither is trained here.

```text
features (2,) -> W @ features (2,) -> add b (2,) -> output (2,)
[0.4, 0.8]                                        [1.3, 0.8]
```

> **Visual to add — One card, two reading strips**
>
> Draw the input card with angle 0.4 and speed 0.8. Lay two transparent strips over the same entries: row one weights 2.0 and 0.5, then adds 0.1; row two weights -0.5 and 1.5, then adds -0.2. Show their contributions meeting at outputs 1.3 and 0.8. Do not route the first row's output into the second row: these recipes run in parallel, not in sequence.
>
> **Proposed caption:** “A matrix lets several questions read the same input.”

In [ ]:
# Part 3: stack two dot-product recipes into one matrix
features = np.array([0.4, 0.8])  # (2,): keep angle and speed as separate input features.

# (2, 2) = (output feature, input feature): stack two recipes so both read the same card.
W = np.array([
    [2.0, 0.5],   # row 1 listens strongly to angle
    [-0.5, 1.5],  # row 2 listens strongly to speed
])
b = np.array([0.1, -0.2])

# (2, 2) @ (2,) -> (2,); b=(2,) adds one offset to each output feature.
output = W @ features + b

print(f"Input vector [angle, speed]: {features}")
print("\nRun each row as one dot product:")
for row_index, (row, bias, result) in enumerate(zip(W, b, output), start=1):
    # (2,) * (2,) keeps the two contributions separate so we can explain each row's sum.
    angle_part, speed_part = row * features
    print(
        f"  row {row_index}: angle {angle_part:+.2f}, speed {speed_part:+.2f}, "
        f"bias {bias:+.2f} → result {result:.2f}"
    )

print(f"\nStacked matrix result: {output.round(3)}")
print("One row gives one output; two rows give an output vector of length two.")

### Why those outputs are 1.3 and 0.8

- First row: `2*0.4 + 0.5*0.8 + 0.1 = 1.3`.
- Second row: `-0.5*0.4 + 1.5*0.8 - 0.2 = 0.8`.

Negative weights subtract a contribution; a bias shifts a result even when the inputs are zero. The outputs are new features, not predicted metres, speed, or scoring probabilities.

`zip(W, b, output)` in the print loop pairs each recipe with its own offset and answer. It explains each row after the vectorized multiplication has already done the work.

### The layer shorthand now has a concrete meaning

`output = W @ features + b` says: read the same feature card with every row, then add one offset to each answer. In maths this is often written `y = Wx + b`; here `x` is the input vector and `y` the output vector, not a known training label.

**Small experiment:** keep `W` and `b` fixed and change only angle from `0.4` to `0.8`. The output change must be `[2*0.4, -0.5*0.4] = [0.8, -0.2]`. Only one input changed; the first column tells you both consequences.

One more complaint remains. So far we have behaved as if the player can reproduce exactly the same kick. What if “aim at 19.11°” produces slightly different angles every time?

## Part 4 · One perfect kick is not a reliable player

Our chosen angle hits one back-net point. A player repeating that intention produces a **spread of executed angles** instead. The question becomes “how much of that spread still clears the wall and enters the goal?”

The next cell scans 20,000 angles between 5° and 60°, keeping those that clear the wall and fit through the goal opening. The sampled legal interval is approximately `18.44° to 21.86°`. Its endpoints are grid estimates, not exact roots.

Now compare two aims:

- `19.11°`: right for the particular back-net target.
- About `20.15°`: halfway between the sampled legal limits.

The second aim leaves equal room for an upward or downward execution error. It need not hit our original back-net point; it targets **any legal score**.

> **Visual to add — Move the aim or tighten the cloud**
>
> Draw a fixed horizontal angle ruler with legal limits 18.44° and 21.86°. Above it compare equal-width bell curves centred at 19.11° and 20.15°, shading only their area inside the limits. Below, hold 20.15° fixed and contrast spreads of 1° and 3°. Do not move the legal interval with the aim or draw a single successful trajectory as proof of the shaded probability.
>
> **Proposed caption:** “Changing aim shifts the cloud. Better consistency narrows it.”

**Predict:** if both players have the same symmetric spread, which aim leaves more of the cloud inside the interval? The centred one. The advantage here is modest, not a miracle shot.

We model the executed angle with a bell-shaped **Gaussian distribution**. `loc` is the intended angle; `scale = sigma = 3°` is its standard deviation, a spread measure rather than a maximum error. Errors beyond 3° are allowed.

If `Theta` names the uncertain angle, `P(theta_lo < Theta < theta_hi)` means the share of this assumed distribution inside the legal window. A cumulative distribution function, `cdf(a)`, gives the share at or below angle `a`. Subtract the shares at the two limits to obtain the scoring probability.

### Read the probability cell as a model calculation

`scoreable_angles` constructs the legal interval. `probability_of_scoring` places a Gaussian over it and subtracts two cumulative probabilities. No random kicks are sampled in this cell.

“Expected scores per 100” means `100 * probability`, not a promise of that many goals in the next match. The calculation includes no goalkeeper, reactions, side-to-side placement, or variation in launch speed. It answers the narrower question we actually modeled.

In [ ]:
# Part 4: compare one point-target aim with a reliable repeated-attempt aim
# Keep angles that clear the wall and cross fully inside the goal frame
scoreable_angles = [
    angle for angle in np.linspace(5, 60, 20_000)
    if (
        ball_height(WALL_X, angle) > WALL_H + BALL_RADIUS
        and GOAL_BOTTOM < ball_height(GOAL_X, angle) < GOAL_TOP
    )
]

if not scoreable_angles:
    raise RuntimeError("No launch angle satisfies the physical constraints")

theta_lo = min(scoreable_angles)
theta_hi = max(scoreable_angles)
point_target_aim = launch_angle_deg
cluster_centered_aim = (theta_lo + theta_hi) / 2
sigma = 3.0


def probability_of_scoring(intended_angle, execution_sigma=sigma):
    distribution = stats.norm(loc=intended_angle, scale=execution_sigma)
    return distribution.cdf(theta_hi) - distribution.cdf(theta_lo)


probability_point_target = probability_of_scoring(point_target_aim)
probability_cluster_centered = probability_of_scoring(cluster_centered_aim)
nll_point_target = -np.log(probability_point_target)
nll_cluster_centered = -np.log(probability_cluster_centered)

print(f"Legal angle window: [{theta_lo:.2f}°, {theta_hi:.2f}°]")
print(f"Typical execution spread: {sigma:.1f}°\n")
print("ONE EXACT INPUT VERSUS A DISTRIBUTION")
print(
    f"  Aim at one chosen point: {point_target_aim:5.2f}°  "
    f"expected scores per 100={100 * probability_point_target:4.1f}"
)
print(
    f"  Center the whole cluster: {cluster_centered_aim:5.2f}°  "
    f"expected scores per 100={100 * probability_cluster_centered:4.1f}"
)
print(f"  Extra expected scores per 100: {100 * (probability_cluster_centered - probability_point_target):+.1f}")
print(f"  Surprise penalty: {nll_point_target:.3f} → {nll_cluster_centered:.3f}")
print("\nPrediction check: answer (b) — centering the cluster leaves more room for imperfect kicks.\n")

print("Consistency check at the cluster-centered aim:")
for spread in [1.0, 2.0, 3.0, 5.0, 8.0, 10.0]:
    probability = probability_of_scoring(cluster_centered_aim, spread)
    print(f"  typical spread={spread:4.1f}°: expected scores per 100={100 * probability:4.1f}")

### Read the two comparisons separately

At a 3° spread, the model expects roughly 41 scores per 100 attempts for the point-target aim and 43 for the centred aim. These are **analytical expectations under our assumptions**, not a simulated tournament.

Then the code holds the centred aim fixed and changes only spread. A 1° standard deviation gives about 91 expected scores per 100; 3° gives about 43; 8° gives about 17. The legal window never changes.

The last extra number is a **surprise penalty**: `-log(p)`. For an event assigned probability `0.9`, the penalty is about `0.105`; for probability `0.1`, it is about `2.303`. Making the event less likely makes observing it more surprising. The logarithm lets independent event probabilities, which multiply, become additive penalties.

Here the event is “a legal score.” This is not the likelihood of a particular observed angle. Classification later uses the same negative-log idea for the probability assigned to the observed class.

## The short version

- A vector stores several related quantities; its entries need names and units.
- A dot product measures a signed projection, not success probability.
- A derivative reads change at one point. Integration accumulates changes across an interval.
- Our 28 exact constant-gravity updates agree with the direct flight formula; more drawing points do not make this physics more accurate.
- A matrix is a stack of dot-product recipes. Learned layers use that operation with adjustable coefficients.
- A point prediction and an outcome distribution answer different questions. Aiming well and being consistent are separate improvements.

**What we actually covered**

- **Calculated and checked:** launch geometry, directional projection, local flight updates, the matching endpoint, a two-output feature transformation, and Gaussian scoring expectations.
- **Talked through:** derivative/integral notation, a state-transition matrix, and negative-log surprise.
- **Left out:** fitting weights, multivariable gradients, and real aerodynamics.

**Try one change:** call `probability_of_scoring(cluster_centered_aim, 1.0)` and then with `8.0`. Keep the aim fixed. The wider cloud should lose probability outside the same legal interval.

The maths gave us ways to describe and compare predictions. It did not tell us how to judge a model trained on examples. [ML Basics](../02-ml-basics/ml-basics.ipynb) changes the scene from a kick to SmartVal's rental listings, where “it fits the examples” is only the start.